In [1]:
import os
import joblib
import pandas as pd
from trainingTools import getbest

In [2]:
ROUTE_DATA=r"C:\Users\Gabo\Downloads\models\models"
paths=os.listdir(ROUTE_DATA)
pathsPooling=[x for x in paths  if 'pooling' in x]
pathsBatch=[x for x in paths if 'seed' in x]
paths_level=[x for x in pathsBatch if 'level' in x]
paths_skill=[x for x in pathsBatch if 'skill' in x]
paths_subject=[x for x in pathsBatch if 'subject' in x]
paths_claridad=set(pathsBatch).difference(
    set(paths_level).union(paths_skill).union(paths_subject)
)

final_level=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_level])

final_subject=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_subject])

final_skill=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_skill])

final_claridad=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_claridad])

In [3]:
#parametros  para  obtener el pooling
groupcols=['pooling']
metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

pooling={}
for x in pathsPooling:
    db=pd.read_csv(f"{ROUTE_DATA}/{x}")
    cabezal=x.split('_')[1]
    best=getbest(db, groupcols,metrics)
    pooling[cabezal]=best['pooling']

print(pooling)

train_f1 :  0.8129072862198964
val_f1 :  0.7278337661742931
0.08507352004560331
{'claridad': 'mean', 'level': 'mean', 'skill': 'mean', 'subject': 'mean'}


In [4]:
groupcols=['num_hidden_layers',
       'hidden_dim', 'activation', 'normalization', 'dropout']

metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

a0=final_level[groupcols+metrics+['seed']]

a=a0[final_level['epoch']==final_level['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [5]:
names=['level', 'skill','subject','claridad']
bases=[final_level,final_skill, final_subject, final_claridad]
info=dict(zip(names,bases))
params={}
for i,j in info.items():
    param=getbest(j,groupcols,metrics)
    print(i)
    print(param['train_f1'],param['val_f1'])
    pool=pooling[i]
    param['pooling']=pool
    params[i]=param
joblib.dump(params,'./finalCabezalParams.joblib')

level
0.9724864315861129 0.9300185014280814
train_f1 :  0.898322539818951
val_f1 :  0.7237118936325939
0.17461064618635713
skill
0.898322539818951 0.7237118936325939
subject
0.9981029466953156 0.9853864323226328
claridad
1.0 0.9985212792932189


['./finalCabezalParams.joblib']

In [6]:
params

{'level': {'num_hidden_layers': np.int64(1),
  'hidden_dim': np.int64(256),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9724864315861129),
  'val_f1': np.float64(0.9300185014280814),
  'train_accuracy': np.float64(0.973775433308214),
  'val_accuracy': np.float64(0.933778715424285),
  'no_overfiting': np.True_,
  'pooling': 'mean'},
 'skill': {'num_hidden_layers': np.int64(2),
  'hidden_dim': np.int64(512),
  'activation': 'gelu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.898322539818951),
  'val_f1': np.float64(0.7237118936325939),
  'train_accuracy': np.float64(0.8945439165374313),
  'val_accuracy': np.float64(0.7173245614035088),
  'no_overfiting': np.False_,
  'pooling': 'mean'},
 'subject': {'num_hidden_layers': np.int64(3),
  'hidden_dim': np.int64(128),
  'activation': 'silu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9

In [9]:
a0=final_skill[groupcols+metrics+['seed']]

a=a0[final_skill['epoch']==final_skill['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [10]:
a=final_skill.copy()
b=a[a['epoch']==a['best_epoch']]
b1=b.groupby(groupcols)[metrics+['train_loss','val_loss']].agg('mean').reset_index()
b1.sort_values(['train_accuracy'], ascending=False)


,num_hidden_layers,hidden_dim,activation,normalization,dropout,train_f1,val_f1,train_accuracy,val_accuracy,train_loss,val_loss
192,2,512,gelu,batchnorm,0.0,0.898323,0.723712,0.894544,0.717325,0.232436,0.708380
272,3,512,relu,batchnorm,0.0,0.897173,0.711179,0.893322,0.703728,0.234185,0.746110
264,3,512,gelu,batchnorm,0.0,0.895220,0.712903,0.892100,0.707237,0.230938,0.740774
280,3,512,silu,batchnorm,0.0,0.892350,0.716599,0.888623,0.714474,0.239707,0.732310
120,1,512,gelu,batchnorm,0.0,0.891617,0.723491,0.888153,0.718640,0.279461,0.651080
...,...,...,...,...,...,...,...,...,...,...,...
49,0,512,gelu,batchnorm,0.1,0.621915,0.610293,0.639598,0.630702,1.075642,1.088741
16,0,128,silu,batchnorm,0.0,0.623548,0.612106,0.639598,0.632895,1.066729,1.080243
3,0,128,gelu,batchnorm,0.5,0.623538,0.620638,0.639551,0.635307,1.065567,1.077646
66,0,512,silu,batchnorm,0.3,0.624881,0.616862,0.639175,0.632675,1.066001,1.077800
